# MVP Demo Runbook — 2026-05-29 (with 2026-06-30 comparison)

This notebook runs the demo case **step by step**. It does not replace the PPT; it produces the numbers the PPT walks through.

Order:

1. **Market regime** — UMD / Daniel–Moskowitz context (deterministic)
2. **Quant signals** — scorecard values, thresholds, trigger states (deterministic)
3. **Structural & mechanical unwind** — mechanism scenarios, concentration, footprint (deterministic)
4. **AI evidence layer** — interpretation of the deterministic layers + evidence challenge
5. **Final PM read** — the combined decision-support read

Default run is **live DeepSeek** (`USE_LLM=True`). Missing key / HTTP failure / schema validation still fail closed to deterministic text. Set `USE_LLM=False` in the setup cell for the fully offline deterministic run.

## Runbook map

| Step | Layer | What you see |
| --- | --- | --- |
| 1 | Market regime | UMD/DM state, drawdown/recovery/volatility conditions, state-conditioned history |
| 2 | Quant signals | 4 deterministic scorecard indicators: value, threshold, status, change vs comparison |
| 3 | Structural / mechanical | 3 mechanism scenarios, theme concentration, factor footprint, turnover, absorption |
| 4 | AI evidence layer | Narrative interpretation + supporting / unconfirmed / premature evidence |
| 5 | Final PM read | One integrated read: current state, vulnerability, why not act yet, next checks |
| 6 | 5/29 vs 6/30 | Same pipeline, two dates: mechanism, fragility, evidence, LLM read |

Technical modes, versions, and cache boundaries are in the appendix at the bottom.

In [1]:
%run -i demo_setup.py


In [2]:
# Run the full MVP once; every step below only renders its layer.
def run_with_retry(cfg, tries=3):
    """Retry the LLM narrative calls; deterministic metrics are identical on every attempt."""
    last = None
    for _ in range(tries):
        last = run_mvp(cfg, interpreter=evidence_interpreter, pm_interpreter=pm_interpreter)
        if last.interpretation.use_llm and last.pm_response.use_llm:
            break
    return last

result = run_with_retry(CONFIG)

evidence = result.deterministic_input
unwind = result.unwind
mechanical = result.mechanical_unwind
interpretation = result.interpretation
pm = result.pm_response

positioning = build_positioning_snapshot(
    as_of_date=CONFIG.as_of_date,
    context_elevated=bool(evidence.triggered_quant_signals),
    processed_dir=CONFIG.processed_dir,
)
positioning_proxies = public_positioning_proxy_items(positioning)

print("MVP demo run complete —", CONFIG.as_of_date, "| fingerprint:", result.full_run_fingerprint)

MVP demo run complete — 2026-05-29 | fingerprint: cae6c2819e643ed7


## Step 1 — Market regime (deterministic)

UMD / Daniel–Moskowitz state and market conditions are **comparison context only**. They are never merged into a PM-book crash probability.

In [3]:
primary = build_primary_assessment(
    as_of_date=pd.Timestamp(CONFIG.as_of_date),
    horizon=CONFIG.horizon_days,
    processed_dir=CONFIG.processed_dir,
)
factors = pd.read_parquet(CONFIG.processed_dir / "french_research_factors_daily.parquet")
regime = build_regime_history(factors)
row = regime.loc[regime["date"].eq(pd.Timestamp(CONFIG.as_of_date))].iloc[0]

rows = [
    ("UMD / DM state", primary.state, "comparison context only"),
    ("Bear state (504d market return < 0)", fmt(row.get("bear_state")), ""),
    ("Market return, 504d", fmt(row.get("mkt_return_504d")), ""),
    ("Market drawdown", fmt(row.get("market_drawdown")), ""),
    ("Recent min drawdown, 126d", fmt(row.get("recent_min_drawdown_126d")), ""),
    ("Recovery from trough, 126d", fmt(row.get("recovery_from_trough_126d")), ""),
    ("High volatility, 21d", fmt(row.get("high_volatility")), ""),
    ("High-vol recovery state", fmt(row.get("high_volatility_recovery_state")), ""),
    ("Rate regime", fmt(row.get("rate_regime")), ""),
]
table = "| Field | Value | Note |\n| --- | --- | --- |\n"
for label, value, note in rows:
    table += f"| {label} | `{value}` | {note} |\n"

analog_rows = "| State | Sample | Tail-loss freq | Mean fwd return | 5th pct |\n| --- | --- | --- | --- | --- |\n"
for item in evidence.historical_analogs:
    analog_rows += (
        f"| `{item['state']}` | {item['sample_size']} | "
        f"{fmt(item['tail_loss_frequency'])} | {fmt(item['mean_forward_return'])} | "
        f"{fmt(item['fifth_percentile_forward_return'])} |\n"
    )

display(Markdown(f"### Market regime — {CONFIG.as_of_date}\n\n{table}"))
display(Markdown(f"**UMD comparison context** (descriptive history, not PM-book probability):\n\n{analog_rows}"))

### Market regime — 2026-05-29

| Field | Value | Note |
| --- | --- | --- |
| UMD / DM state | `normal` | comparison context only |
| Bear state (504d market return < 0) | `False` |  |
| Market return, 504d | `0.4863` |  |
| Market drawdown | `0.0000` |  |
| Recent min drawdown, 126d | `-0.0946` |  |
| Recovery from trough, 126d | `0.1948` |  |
| High volatility, 21d | `False` |  |
| High-vol recovery state | `False` |  |
| Rate regime | `tightening` |  |


**UMD comparison context** (descriptive history, not PM-book probability):

| State | Sample | Tail-loss freq | Mean fwd return | 5th pct |
| --- | --- | --- | --- | --- |
| `all` | 25629 | 0.0505 | 0.0054 | -0.0587 |
| `normal` | 21132 | 0.0336 | 0.0075 | -0.0464 |
| `bear_low_volatility` | 3116 | 0.0815 | -0.0012 | -0.0791 |
| `panic_elevated` | 1381 | 0.2390 | -0.0130 | -0.1817 |


## Step 2 — Quant signals (deterministic scorecard)

Four indicators with explicit values and thresholds. Status is **triggered / not triggered**, not a probability.

In [4]:
signals = list(evidence.triggered_quant_signals) + list(evidence.non_triggered_relevant_signals)
rows = []
for s in signals:
    delta = fmt(s.change_vs_comparison, signed=True) if s.change_vs_comparison is not None else "—"
    rows.append(
        f"| `{s.name}` | `{fmt(s.current_value)}` | `{fmt(s.threshold)}` | "
        f"`{s.status}` | `{delta}` | {s.interpretation} |"
    )
table = (
    "| Metric | Value | Threshold | Status | Δ vs comparison | Read |\n"
    "| --- | --- | --- | --- | --- | --- |\n"
    + "\n".join(rows)
)
display(Markdown(f"### Quant signals — {CONFIG.as_of_date}\n\n{table}"))

### Quant signals — 2026-05-29

| Metric | Value | Threshold | Status | Δ vs comparison | Read |
| --- | --- | --- | --- | --- | --- |
| `high_volatility_recovery` | `0.0000` | `1.0000` | `not_triggered` | `+0.0000` | One composite macro gate replaces separate drawdown, recovery, and volatility alerts. It requires Phase 1 early recovery and high realized volatility to be true together. |
| `short_minus_long_beta_gap` | `-2.0677` | `0.2490` | `not_triggered` | `-0.1726` | Positive and unusually high short-underlying minus long beta indicates that a market rebound can squeeze the recent-loser leg. Threshold: prior-only 80th percentile from 2281 observations; raw historical threshold=0.248979. |
| `portfolio_drawdown` | `-0.0752` | `-0.1741` | `not_triggered` | `-0.0752` | Long-short wealth relative to its highest level in the prior 63 trading days is compared with its own prior-only left-tail history. The threshold can never be looser than -20%, and drawdowns shallower than 5% are not material. Threshold: prior-only 20th percentile from 2281 observations; raw historical threshold=-0.174081. |
| `short_loss_in_recovery` | `0.1551` | `0.2504` | `not_triggered` | `+0.0046` | Trailing 21-day short loss magnitude is the sum of negative signed short contributions. It triggers only when Phase 1 early recovery is active and the loss reaches the threshold. Threshold: prior-only 80th percentile from 2323 observations; raw historical threshold=0.25042. |

## Step 3 — Structural & mechanical unwind (deterministic)

Three independent mechanism lenses, then concentration and market-footprint proxies. These layers are read separately from the quant scorecard.

In [5]:
mech_rows = []
for m in unwind.mechanism_scenarios:
    mech_rows.append(
        f"| `{m.scenario}` | `{m.status}` | {m.summary} |"
    )
mech_table = (
    "| Mechanism scenario | Status | Read |\n"
    "| --- | --- | --- |\n"
    + "\n".join(mech_rows)
)

tc = unwind.theme_concentration
theme_rows = [
    ("Cluster", ", ".join(tc.cluster_symbols) or "—"),
    ("Active long symbols", ", ".join(tc.active_long_symbols) or "—"),
    ("Cluster exposure share", fmt(tc.cluster_exposure_share)),
    ("Avg residual correlation", fmt(tc.cluster_average_residual_correlation)),
    ("5d residual loss", fmt(tc.cluster_residual_loss_5d)),
    ("5d abnormal volume share", fmt(tc.cluster_abnormal_volume_share_5d)),
    ("Concentration trigger", fmt(tc.trigger)),
]
theme_table = "| Field | Value |\n| --- | --- |\n"
for label, value in theme_rows:
    theme_table += f"| {label} | `{value}` |\n"

mech_state_rows = [
    ("Unwind state", fmt(mechanical.unwind_state)),
    ("Control spec", fmt(mechanical.control_spec)),
    ("Factor footprint R²", fmt(mechanical.factor_footprint_r2)),
    ("Factor footprint percentile", fmt(mechanical.factor_footprint_percentile)),
    ("Extreme turnover ratio", fmt(mechanical.extreme_turnover_ratio)),
    ("Extreme turnover percentile", fmt(mechanical.extreme_turnover_percentile)),
    ("Liquidity absorption failure", fmt(mechanical.liquidity_absorption_failure)),
    ("Absorption percentile", fmt(mechanical.absorption_percentile)),
]
mech_state_table = "| Field | Value |\n| --- | --- |\n"
for label, value in mech_state_rows:
    mech_state_table += f"| {label} | `{value}` |\n"

sc_rows = []
for r in unwind.scorecard:
    sc_rows.append(
        f"| `{r.metric}` | `{fmt(r.current_value)}` | `{fmt(r.threshold)}` | "
        f"`{r.triggered}` | `{r.severity}` | {r.explanation} |"
    )
sc_table = (
    "| Metric | Value | Threshold | Triggered | Severity | Explanation |\n"
    "| --- | --- | --- | --- | --- | --- |\n"
    + "\n".join(sc_rows)
)

display(Markdown(f"### Mechanism scenarios — {CONFIG.as_of_date}\n\n{mech_table}"))
display(Markdown(f"### Theme concentration\n\n{theme_table}"))
display(Markdown(f"### Mechanical footprint\n\n{mech_state_table}"))
display(Markdown(f"### Unwind scorecard (6 rows)\n\n{sc_table}"))

### Mechanism scenarios — 2026-05-29

| Mechanism scenario | Status | Read |
| --- | --- | --- |
| `bear_market_recovery_crash` | `watch` | Some bear-market-recovery preconditions are present, but the three-part mechanism is not confirmed. |
| `short_book_reversal_crash` | `not_confirmed` | Short-book reversal conditions are not present. |
| `crowded_theme_unwind` | `triggered` | A pre-event correlated long cluster is concentrated and is experiencing broad, extreme, loss- or volume-confirmed selling. |

### Theme concentration

| Field | Value |
| --- | --- |
| Cluster | `CIEN, COHR, LITE` |
| Active long symbols | `CIEN, COHR, ECHO, FIX, LITE, MU, SNDK, STX, TER, WDC` |
| Cluster exposure share | `0.3000` |
| Avg residual correlation | `0.7256` |
| 5d residual loss | `0.0738` |
| 5d abnormal volume share | `0.6667` |
| Concentration trigger | `True` |


### Mechanical footprint

| Field | Value |
| --- | --- |
| Unwind state | `FRAGILITY_BUILDING` |
| Control spec | `mom_vol` |
| Factor footprint R² | `0.1251` |
| Factor footprint percentile | `0.7689` |
| Extreme turnover ratio | `1.1125` |
| Extreme turnover percentile | `0.8566` |
| Liquidity absorption failure | `False` |
| Absorption percentile | `0.2709` |


### Unwind scorecard (6 rows)

| Metric | Value | Threshold | Triggered | Severity | Explanation |
| --- | --- | --- | --- | --- | --- |
| `portfolio_concentration` | `19.5188` | `19.7888` | `True` | `high` | Gross-normalized effective bets use drifted beginning-of-day exposure; lower values indicate greater concentration. |
| `momentum_breadth_deterioration` | `0.6580` | `0.5558` | `False` | `normal` | The share of the eligible universe with positive 12-1 momentum is compared with its strictly prior monthly history. |
| `synchronous_winner_liquidation` | `-0.0059` | `0.0192` | `False` | `normal` | An extreme five-day lagged-beta-adjusted long loss must coincide with broad active-long declines. |
| `cross_sectional_reversal` | `-0.0347` | `0.0331` | `False` | `normal` | Positive short-underlying-minus-long return means prior losers outperformed prior winners over five trading days. |
| `liquidity_amplification_proxy` | `0.3000` | `0.5000` | `False` | `normal` | This public-data proxy counts active long names falling while five-day volume exceeds each name's prior-only threshold. |
| `fundamental_anchor` | `unavailable` | `coverage-gated sign-vote rule` | `None` | `unavailable` | Revenue acceleration, applicable operating-margin change, and optional EPS acceleration provide a lightweight sign-based anchor. |

## Step 4 — AI evidence layer

The AI layer interprets the deterministic layers and organizes supporting / contradicting / missing evidence. It **cannot change** any deterministic value, threshold, or trigger.

In [6]:
mode = "live DeepSeek" if interpretation.use_llm else "offline deterministic"
support_ids = ", ".join(interpretation.supporting_evidence_ids) or "—"
contra_ids = ", ".join(interpretation.contradicting_evidence_ids) or "—"
evidence_quality = evidence.audit_metadata.get("evidence_quality", "unavailable")

warn_block = ""
if evidence.data_warnings:
    warn_block = (
        "<details><summary>Deterministic adapter warnings (technical)</summary>\n\n"
        + bullets(evidence.data_warnings)
        + "\n\n</details>"
    )

display(Markdown(f"""
### AI evidence layer — {mode}

**Narrative state:** {interpretation.narrative_state}

**Interpretation:** “{interpretation.pm_interpretation}”

**Supporting evidence IDs:** {support_ids}  
**Contradicting evidence IDs:** {contra_ids}

**Missing / uncertain evidence:**

{bullets(interpretation.missing_or_uncertain_evidence[:4])}

**Monitoring questions:**

{bullets(interpretation.monitoring_questions)}

**Invalidation conditions:**

{bullets(interpretation.invalidation_conditions)}

*Evidence quality:* `{evidence_quality}` · *Version:* `{interpretation.model_or_prompt_version}`

{warn_block}
"""))

case_read = (CASE_PACKS["current_semi"] / "pm_case_read.md").read_text()
if CONFIG.as_of_date == "2026-05-29":
    display(Markdown("**Frozen evidence challenge (2026-05-29 pack)** — supporting / unconfirmed / premature."))
    for title in [
        "What is supported",
        "What remains unconfirmed",
        "Why broad action may still be premature",
    ]:
        display(Markdown(section(title, case_read)))
else:
    display(Markdown("_The frozen evidence challenge is bound to 2026-05-29 and is not shown for other CONFIG dates._"))


### AI evidence layer — live DeepSeek

**Narrative state:** Crowded momentum unwind scenario active; signals below escalation thresholds; monitoring for factor propagation and liquidity stress.

**Interpretation:** “Quantitative signals remain below escalation thresholds, but the structural crowded-theme unwind is triggered with elevated turnover in the loser basket. Public short-interest proxies are elevated, suggesting short-side crowding is plausible, but this does not establish active covering or forced deleveraging. Fundamental earnings from major tech names are contradicting the unwind narrative, providing support for valuations. The absence of a broad factor footprint and liquidity absorption failure suggests no broad mechanical unwind is confirmed. The setup warrants monitoring for factor propagation and liquidity stress, but no active momentum unwind is indicated.”

**Supporting evidence IDs:** csu-2026-05-29-013  
**Contradicting evidence IDs:** csu-2026-05-29-001, csu-2026-05-29-006, csu-2026-05-29-007, csu-2026-05-29-008, csu-2026-05-29-009, csu-2026-05-29-010, csu-2026-05-29-011, csu-2026-05-29-012

**Missing / uncertain evidence:**

- No direct evidence on leverage, margin, or financing stress; forced deleveraging remains unconfirmed.
- No observed book-level flows; Prime Brokerage commentary is aggregated and does not identify specific investors.
- No direct evidence on common ownership or coordinated selling; correlated price action alone does not establish crowding.

**Monitoring questions:**

- Is the elevated turnover in the loser basket accompanied by a broadening factor footprint?
- Are the elevated short-interest proxies in the loser basket translating into actual covering or further selling?
- Do the fundamental earnings reports from major tech names continue to support valuations, or are there signs of capex repricing?

**Invalidation conditions:**

- If the short-loss-in-recovery signal triggers while the high-volatility recovery signal remains untriggered, the recovery-crash scenario becomes more plausible.
- If the factor footprint becomes elevated while turnover remains elevated, a broad mechanical unwind is more likely.
- If the short-interest utilisation proxy normalizes while the loser basket continues to underperform, short-side crowding as a driver is weakened.

*Evidence quality:* `available` · *Version:* `evidence-interpretation-prompt-v8`

<details><summary>Deterministic adapter warnings (technical)</summary>

- The cached corpus is small and may omit relevant contradictory evidence.
- Generic macro context does not establish momentum-specific causality.
- Evidence cannot change deterministic metrics, thresholds, triggered states, or create a risk score.
- No composite deterministic score is defined; the adapter preserves the four indicator states and leaves deterministic_score null.
- The legacy Phase 6 adapter contract contains Phase 5A feasibility metadata only and does not embed the separate Phase 5 unwind scorecard; the notebook renders that deterministic assessment alongside this card. Its fundamental row remains unavailable unless exact-date company coverage is supplied.

</details>


**Frozen evidence challenge (2026-05-29 pack)** — supporting / unconfirmed / premature.

## What is supported

- Structured crowded-theme unwind and concentration stress in the PM book.
- Elevated turnover and concentrated pressure, with no sign that market liquidity is failing.
- Market-recovery component in retrieved text (`CSU-2026-015`).
- Contradicting operating strength at a cluster name (`CSU-2026-008`).

## What remains unconfirmed

- Forced deleveraging, financing pressure, or dealer-inventory stress.
- Factor propagation beyond the detected cluster.
- Liquidity-absorption failure.
- Complete DM sequence (panic + loser-leg rebound + short-leg loss).
- Completed fundamental valuation or earnings reprice.
- Stance-confirmed citation of contextual items `CSU-2026-013`, `CSU-2026-004`, `CSU-2026-005` (MVP citation limitation).

## Why broad action may still be premature

A crowded-theme signal warrants focused review, but broad automatic de-risking is still premature: selling is being absorbed, stress has not spread beyond the cluster, the recovery-crash and fundamental lenses remain weak, and positioning evidence is not strong enough to confirm the narrative. Any assessment of the later semiconductor selloff requires a refreshed portfolio snapshot and evidence from the same cutoff.

## Step 5 — Final PM read

All layers collapse into one decision-support read: what is happening, where the risk sits, why broad action is premature, and what would change the reading.

In [7]:
scenarios = {m.scenario: m.status for m in unwind.mechanism_scenarios}
tail_risk_state = {
    "FRAGILITY_BUILDING": "potential momentum tail risk",
}.get(str(mechanical.unwind_state), str(mechanical.unwind_state).replace("_", " ").lower())

table = (
    "| Layer | Read |\n"
    "| --- | --- |\n"
    f"| UMD / market context | `{evidence.overall_risk_state}` (comparison only) |\n"
    f"| Scorecard triggers | {len(evidence.triggered_quant_signals)} |\n"
    f"| Recovery crash | `{scenarios.get('bear_market_recovery_crash')}` |\n"
    f"| Short-book reversal | `{scenarios.get('short_book_reversal_crash')}` |\n"
    f"| Crowded theme unwind | `{scenarios.get('crowded_theme_unwind')}` |\n"
    f"| Momentum tail-risk state | `{tail_risk_state}` |\n"
    f"| Classification | `{unwind.scenario_classification}` |\n"
)

cats = [CATEGORY_LABELS.get(c, c) for c in pm.response_categories]
cats_block = (
    "<details><summary>Response categories (bounded menu)</summary>\n\n"
    + bullets(cats)
    + "\n\n</details>"
)
pm_mode = "live DeepSeek" if pm.use_llm else "offline deterministic"

display(Markdown(f"""
### Final PM read — {CONFIG.as_of_date}

{table}
**Current read:** “{pm.current_state}”

**Main vulnerability:** “{pm.main_vulnerability}”

**Why not act yet:** “{pm.why_not_act_yet}”

**What would change the reading:**

{bullets(pm.what_would_change_the_reading)}

**Conditional response:**

{bullets(pm.conditional_response)}

{cats_block}

*Mode: {pm_mode} (`{pm.model_or_prompt_version}`)*
"""))


### Final PM read — 2026-05-29

| Layer | Read |
| --- | --- |
| UMD / market context | `normal` (comparison only) |
| Scorecard triggers | 0 |
| Recovery crash | `watch` |
| Short-book reversal | `not_confirmed` |
| Crowded theme unwind | `triggered` |
| Momentum tail-risk state | `potential momentum tail risk` |
| Classification | `crowded_momentum_unwind` |

**Current read:** “No deterministic escalation signals are active today; the crowded-theme unwind mechanism is triggered, but the short-book reversal crash is not confirmed, and the overall risk state remains normal.”

**Main vulnerability:** “The primary vulnerability is rebound-sensitive shorts in the momentum-loser basket, supported by elevated short-interest ratio and utilization proxies in the public data universe, though this path is not active today; the confirmed stress is long-side crowding, so we are watching for any shift toward short-side recovery risk.”

**Why not act yet:** “We are not taking further action because the short-side recovery risk is not confirmed; the elevated short-interest proxies are contextual but not proof of covering or forced deleveraging, and the deterministic signals for a short-book reversal are not triggered, so maintaining posture and monitoring is appropriate.”

**What would change the reading:**

- A confirmed trigger of the short-book reversal crash mechanism would change the reading, as would a short-loss-in-recovery signal during a recovery regime.
- A widening short-minus-long beta gap into a triggered state or a portfolio drawdown breaching its threshold would also prompt a reassessment.
- If the short-interest proxy states shift further (e.g., utilization rising while volume share remains depressed), that would support the crowding hypothesis and warrant a closer look.

**Conditional response:**

- If the short book reversal crash mechanism moves from watch to triggered, we would escalate to a formal PM review and consider reducing short exposure subject to your approval.
- If short losses in a recovery regime trigger, we would run a loser-rally stress scenario and review rebound-sensitive shorts.
- If the short-minus-long beta gap widens into a triggered state, we would consider a temporary beta hedge subject to PM review.

<details><summary>Response categories (bounded menu)</summary>

- Maintain the current posture and monitor the short basket
- Check whether short concentration is amplifying the move
- Run a loser-rally stress scenario

</details>

*Mode: live DeepSeek (`pm-response-prompt-v5`)*


## Step 6 — 2026-05-29 vs 2026-06-30 (same pipeline)

Same rules, two dates, both with exact-date evidence replay and live DeepSeek interpretation. The point is not a forecast: it shows how the mechanism read and the fragility footprint changed between the two cutoffs.

In [8]:
# Run 2026-06-30 through the identical pipeline (live DeepSeek).
config_630 = MVPConfig(
    as_of_date="2026-06-30",
    compare_to_date="2026-05-29",
    threshold_profile="default",
    horizon_days=20,
    use_llm=USE_LLM,
)
result_630 = run_with_retry(config_630)

ev6 = result_630.deterministic_input
un6 = result_630.unwind
mech6 = result_630.mechanical_unwind
interp6 = result_630.interpretation
pm6 = result_630.pm_response

def posture_for(unw):
    return "escalate_for_pm_review" if "crowded_theme_unwind" in unw.active_scenarios else "focused_review_and_monitor"

rows = [
    ("Overall state", evidence.overall_risk_state, ev6.overall_risk_state),
    ("Quant triggers", len(evidence.triggered_quant_signals), len(ev6.triggered_quant_signals)),
    ("Scenario classification", unwind.scenario_classification, un6.scenario_classification),
    ("Active mechanisms", ", ".join(unwind.active_scenarios) or "none", ", ".join(un6.active_scenarios) or "none"),
    ("Theme cluster", ", ".join(unwind.theme_concentration.cluster_symbols) or "—", ", ".join(un6.theme_concentration.cluster_symbols) or "—"),
    ("Theme trigger", fmt(unwind.theme_concentration.trigger), fmt(un6.theme_concentration.trigger)),
    ("Mechanical state", mechanical.unwind_state, mech6.unwind_state),
    ("Factor footprint pct", fmt(mechanical.factor_footprint_percentile), fmt(mech6.factor_footprint_percentile)),
    ("Extreme turnover pct", fmt(mechanical.extreme_turnover_percentile), fmt(mech6.extreme_turnover_percentile)),
    ("Liquidity absorption failure", fmt(mechanical.liquidity_absorption_failure), fmt(mech6.liquidity_absorption_failure)),
    ("Evidence items", evidence.audit_metadata["retrieved_evidence_count"], ev6.audit_metadata["retrieved_evidence_count"]),
    ("Evidence mode", "live DeepSeek" if interpretation.use_llm else "deterministic", "live DeepSeek" if interp6.use_llm else "deterministic"),
    ("LLM narrative", interpretation.narrative_state, interp6.narrative_state),
    ("PM posture", posture_for(unwind), posture_for(un6)),
]
table = "| Dimension | 2026-05-29 | 2026-06-30 |\n| --- | --- | --- |\n" + "\n".join(f"| {k} | {v5} | {v6} |" for k, v5, v6 in rows)
display(Markdown(f"### What changed from 2026-05-29 to 2026-06-30\n\n{table}"))


### What changed from 2026-05-29 to 2026-06-30

| Dimension | 2026-05-29 | 2026-06-30 |
| --- | --- | --- |
| Overall state | normal | normal |
| Quant triggers | 0 | 0 |
| Scenario classification | crowded_momentum_unwind | normal_drawdown |
| Active mechanisms | crowded_theme_unwind | none |
| Theme cluster | CIEN, COHR, LITE | MU, STX, WDC |
| Theme trigger | True | False |
| Mechanical state | FRAGILITY_BUILDING | FRAGILITY_BUILDING |
| Factor footprint pct | 0.7689 | 0.8606 |
| Extreme turnover pct | 0.8566 | 0.0040 |
| Liquidity absorption failure | False | True |
| Evidence items | 15 | 3 |
| Evidence mode | live DeepSeek | live DeepSeek |
| LLM narrative | Crowded momentum unwind scenario active; signals below escalation thresholds; monitoring for factor propagation and liquidity stress. | Normal drawdown; no confirmed escalation; potential momentum tail risk. |
| PM posture | escalate_for_pm_review | focused_review_and_monitor |

## Cross-case comparison (one screen)

Same rules, three different conclusions. Values come from repository outputs, not hard-coded demo claims.

In [9]:
display(Markdown((CASE_PACKS["cross_case"]).read_text()))

# Cross-case comparison

Derived from repository case packs and `run_mvp` outputs. Mechanism labels are descriptive reads, not crash probabilities.

| Question | Current semi case (2026-05-29) | 2020 validation (2020-03-24) | 2024 quiet control (2024-01-05) |
| --- | --- | --- | --- |
| Recovery mechanism (Daniel–Moskowitz) | Partial / watch — recovery text without panic, loser rebound, or short-loss confirmation | Strongly present — `bear_market_recovery_crash` triggered; panic, severe drawdown, high vol, recovery aligned | Not present as a completed setup — recovery precondition only; severe drawdown and high vol unmet |
| Crowded unwind evidence (Khandani–Lo) | Contextual / partially supported — `crowded_theme_unwind` triggered; concentration and potential momentum tail risk, while selling is still being absorbed | Secondary / unconfirmed — liquidity facilities ≠ crowded positioning | Limited — crowded-theme scenario not confirmed; trading footprint is normal |
| Short-leg pressure | Contained on scorecard; risk sits more in long-side crowding | Severe — `short_loss_in_recovery` and beta-gap triggered; short-book reversal on watch | Contained — short-loss, beta-gap, and short-book reversal not triggered |
| Evidence confidence | Mixed — localized crowding supported; broad crash and forced unwind unconfirmed | Historically coherent — mechanism indicators line up with a known reversal episode | Low-risk / quiet — ordinary macro context; no confirmed crash channel |
| PM workflow | Monitor and investigate concentrated / theme exposures | Escalate review of recovery-crash and short-basket channels | Maintain monitoring — escalation not justified |

## How to read the table

1. **Current semi** is the primary live-style product demo: localized crowding pressure without a completed recovery crash.
2. **2020** shows that when a historically important momentum reversal occurred, the recovery-crash indicators behaved coherently.
3. **2024** shows the same rules staying quiet when the mechanism is incomplete.

Sources:

- `outputs/current_semi_unwind/pm_case_read.md`
- `outputs/current_semi_unwind/mechanism_comparison.md`
- `outputs/march_2020_reference/pm_case_read.md`
- `outputs/march_2020_reference/mechanism_comparison.md`
- `outputs/quiet_control_2024/pm_case_read.md`
- `outputs/quiet_control_2024/mechanism_comparison.md`
- `outputs/research_validation/episode_fingerprints.md`


## Case packs and references

The runbook covers **two momentum-crash mechanisms** — **Daniel–Moskowitz** recovery-driven reversal and **Khandani–Lo** crowded-position unwind — and the **AI evidence view** in Step 4.

Frozen/reference packs used by the demo:

| Pack | Path |
| --- | --- |
| Primary correlated-cluster case (2026-05-29) | `current_semi_unwind` → `outputs/snapshot_2026-05-29` |
| 2020 historical validation | `march_2020_reference` |
| 2024 quiet control | `quiet_control_2024` |
| Cross-case comparison | `cross_case_comparison.md` |
| Production path | `docs/production_path.md` |

## Case packs and references

The runbook covers **Two momentum-crash mechanisms** — **Daniel–Moskowitz** recovery-driven reversal and **Khandani–Lo** crowded-position unwind — and the **AI evidence view** in Step 4.

Frozen/reference packs used by the demo:

| Pack | Path |
| --- | --- |
| Primary correlated-cluster case (2026-05-29) | `current_semi_unwind` → `outputs/snapshot_2026-05-29` |
| 2020 historical validation | `march_2020_reference` |
| 2024 quiet control | `quiet_control_2024` |
| Cross-case comparison | `cross_case_comparison.md` |
| Production path | `docs/production_path.md` |

<details>
<summary><strong>Technical appendix — not needed during the demo</strong></summary>

- **Run modes:** This runbook defaults to `USE_LLM=True` (live DeepSeek) for the PPT demo; it requires `DEEPSEEK_API_KEY` in `.env`. Set `USE_LLM=False` for offline deterministic Evidence Card + PM narrative, no API call. Missing key / HTTP failure / schema validation fails closed to deterministic text and never rewrites metrics.
- **Evidence layer:** exact-date validated cache replay via `src/evidence/research_preview.py`; missing, malformed, future-dated, or post-cutoff evidence fails closed to `unavailable`.
- **Evidence cache coverage:** 2026-05-29 now has an exact-date replay of 15 human-reviewed CSU records (hash-verified, cutoff-valid); 2026-06-30 has the bundled 3-item minimal cache. Missing or invalid caches fail closed to `unavailable`, and evidence can never change deterministic fields.
- **Versions:** deterministic evidence interpretation `deterministic-evidence-interpretation-v2`; deterministic PM response `deterministic-pm-response-v1`; live prompts `evidence-interpretation-prompt-v8` / `pm-response-prompt-v5`.
- **Quant components:** `src/mvp/evidence_card.py`, `src/mvp/pipeline.py`, `src/monitoring/scorecard.py`, `src/monitoring/unwind_monitor.py`, `src/regime/market_state.py`.
- **Not a prediction or trade instruction.**

</details>

**Runbook complete.** The PPT presents the story; this notebook produced the step-by-step numbers for the 2026-05-29 case and the 2026-06-30 comparison.